In [ ]:
# ============================================================
# TALTECH AI PIPELINE
# NOTEBOOK 07A — TABULAR REPRESENTATION
# Portable Windows/Linux
#
# Purpose:
# - Build the tabular representation from Notebook 06.
# - Keep the SAME chronological train/validation/test assignment.
# - Create 1-hour history using 4 lags at 15-min resolution.
# - Add cyclical calendar features.
# - Never create lags across a temporal segment boundary.
# - Fit StandardScaler ONLY on training predictors.
# - Keep unscaled datasets too (important for tree models).
#
# Target:
# temperature at t+15 min
#
# INPUT:
# outputs/06_common_experiment/01_common_supervised_dataset.csv
#
# OUTPUT:
# outputs/07A_tabular/
# ============================================================

from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
INPUT_FILE = (
    OUTPUT_DIR
    / "06_common_experiment"
    / "01_common_supervised_dataset.csv"
)
OUT = OUTPUT_DIR / "07A_tabular"
OUT.mkdir(parents=True, exist_ok=True)

print("====================================")
print("NOTEBOOK 07A — TABULAR REPRESENTATION")
print("====================================")
print("Project :", PROJECT_DIR)
print("Input   :", INPUT_FILE)
print("Output  :", OUT)

assert INPUT_FILE.exists(), (
    f"\nInput file not found:\n{INPUT_FILE}\n"
    "Run Notebook 06 first."
)

# ============================================================
# 2. PARAMETERS
# ============================================================

BASE_VARIABLES = [
    "temperature",
    "relative_humidity",
    "co2"
]

N_LAGS = 4
STEP_MINUTES = 15
TARGET = "target_temperature_t_plus_15"

# ============================================================
# 3. LOAD COMMON SUPERVISED DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], utc=True, errors="coerce"
)
df["target_timestamp"] = pd.to_datetime(
    df["target_timestamp"], utc=True, errors="coerce"
)

numeric_cols = BASE_VARIABLES + [TARGET]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

required = {
    "uuid", "timestamp", "segment_id", "split",
    "target_timestamp", TARGET, *BASE_VARIABLES
}

missing = required - set(df.columns)
assert not missing, f"Missing columns: {sorted(missing)}"

assert df["timestamp"].notna().all()
assert df["target_timestamp"].notna().all()
assert df[numeric_cols].notna().all().all()

df = df.sort_values(
    ["uuid", "segment_id", "timestamp"]
).reset_index(drop=True)

print("\nRows loaded :", len(df))
print("Devices     :", df["uuid"].nunique())
print("Splits      :", df["split"].value_counts().to_dict())

# ============================================================
# 4. CREATE LAGS INSIDE EACH CONTIGUOUS SEGMENT
# ============================================================
# For each variable:
# lag_1 = t-15 min
# lag_2 = t-30 min
# lag_3 = t-45 min
# lag_4 = t-60 min
#
# Grouping by UUID + segment_id prevents any lag from crossing
# a long missing-data gap.

group_cols = ["uuid", "segment_id"]

for variable in BASE_VARIABLES:
    for lag in range(1, N_LAGS + 1):
        df[f"{variable}_lag_{lag}"] = (
            df.groupby(group_cols, sort=False)[variable]
            .shift(lag)
        )

# ============================================================
# 5. STRICT LAG-TIMESTAMP VALIDATION
# ============================================================
# Shift inside a segment is necessary but we additionally verify
# that four preceding records correspond exactly to t-15,...,t-60.

for lag in range(1, N_LAGS + 1):
    previous_timestamp = (
        df.groupby(group_cols, sort=False)["timestamp"]
        .shift(lag)
    )

    expected_timestamp = (
        df["timestamp"]
        - pd.Timedelta(minutes=STEP_MINUTES * lag)
    )

    df[f"_lag_{lag}_time_valid"] = (
        previous_timestamp == expected_timestamp
    )

lag_time_valid_cols = [
    f"_lag_{lag}_time_valid"
    for lag in range(1, N_LAGS + 1)
]

strict_history_mask = df[lag_time_valid_cols].all(axis=1)

# ============================================================
# 6. CALENDAR FEATURES
# ============================================================

# UTC calendar variables are used because source timestamps are UTC.
df["hour_decimal"] = (
    df["timestamp"].dt.hour
    + df["timestamp"].dt.minute / 60.0
)

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour_decimal"] / 24.0
)
df["hour_cos"] = np.cos(
    2 * np.pi * df["hour_decimal"] / 24.0
)

df["day_of_week"] = df["timestamp"].dt.dayofweek

df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7.0
)
df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7.0
)

# ============================================================
# 7. DEFINE TABULAR PREDICTORS
# ============================================================

lag_features = [
    f"{variable}_lag_{lag}"
    for variable in BASE_VARIABLES
    for lag in range(1, N_LAGS + 1)
]

current_features = BASE_VARIABLES.copy()

calendar_features = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos"
]

FEATURES = (
    current_features
    + lag_features
    + calendar_features
)

# We intentionally exclude raw UUID from predictors.
# This avoids imposing an arbitrary numeric meaning on device IDs.
# UUID is retained as metadata for evaluation by device.

feature_complete_mask = df[FEATURES].notna().all(axis=1)

tabular = df.loc[
    strict_history_mask & feature_complete_mask
].copy()

# Remove temporary validation columns.
tabular = tabular.drop(
    columns=lag_time_valid_cols
)

print("\n====================================")
print("TABULAR SAMPLE CONSTRUCTION")
print("====================================")
print("Common supervised samples :", len(df))
print("Valid 1-hour histories    :", len(tabular))
print("Removed for lag history   :", len(df) - len(tabular))
print("Predictor count           :", len(FEATURES))

# ============================================================
# 8. VERIFY SPLITS ARE PRESERVED
# ============================================================

split_order = ["train", "validation", "test"]

split_counts = (
    tabular["split"]
    .value_counts()
    .reindex(split_order)
    .fillna(0)
    .astype(int)
)

print("\nSamples after lag construction:")
print(split_counts.to_string())

assert set(tabular["split"].unique()) == set(split_order)

# No lag can use another split because all historical rows precede
# the current sample, but at the START of validation/test we also
# enforce representation purity: all four lag timestamps must lie
# inside the same split as the current row.
#
# This gives each pipeline a self-contained partition and avoids
# using training-period observations as validation predictors.

split_start = (
    tabular.groupby("split")["timestamp"].min().to_dict()
)

history_start = (
    tabular["timestamp"]
    - pd.Timedelta(minutes=N_LAGS * STEP_MINUTES)
)

same_split_history = pd.Series(True, index=tabular.index)

for split_name in split_order:
    mask = tabular["split"] == split_name
    same_split_history.loc[mask] = (
        history_start.loc[mask]
        >= split_start[split_name]
    )

before_purity = len(tabular)
tabular = tabular.loc[same_split_history].copy()
purity_removed = before_purity - len(tabular)

print("Partition-boundary histories removed :", purity_removed)

# ============================================================
# 9. FINAL UN-SCALED TABULAR DATASETS
# ============================================================

metadata_cols = [
    "uuid",
    "timestamp",
    "segment_id",
    "target_timestamp",
    "split"
]

final_cols = metadata_cols + FEATURES + [TARGET]

tabular = (
    tabular[final_cols]
    .sort_values(["timestamp", "uuid"])
    .reset_index(drop=True)
)

train = tabular.loc[tabular["split"] == "train"].copy()
val = tabular.loc[tabular["split"] == "validation"].copy()
test = tabular.loc[tabular["split"] == "test"].copy()

assert len(train) > 0 and len(val) > 0 and len(test) > 0

# ============================================================
# 10. TRAIN-ONLY STANDARDIZATION
# ============================================================
# StandardScaler is useful for Ridge and provides a reproducible
# scaled representation. RF/XGBoost may use the unscaled files.
#
# IMPORTANT: fit ONLY on TRAIN.

scaler = StandardScaler()
scaler.fit(train[FEATURES])

def make_scaled(frame):
    out = frame.copy()
    out.loc[:, FEATURES] = scaler.transform(frame[FEATURES])
    return out

train_scaled = make_scaled(train)
val_scaled = make_scaled(val)
test_scaled = make_scaled(test)

with (OUT / "tabular_standard_scaler.pkl").open("wb") as f:
    pickle.dump(scaler, f)

scaler_stats = pd.DataFrame({
    "feature": FEATURES,
    "train_mean": scaler.mean_,
    "train_scale": scaler.scale_
})

scaler_stats.to_csv(
    OUT / "07_scaler_train_statistics.csv",
    index=False
)

# ============================================================
# 11. SAVE DATASETS
# ============================================================

tabular.to_csv(
    OUT / "01_tabular_all_unscaled.csv",
    index=False
)

train.to_csv(
    OUT / "02_train_tabular_unscaled.csv",
    index=False
)
val.to_csv(
    OUT / "03_validation_tabular_unscaled.csv",
    index=False
)
test.to_csv(
    OUT / "04_test_tabular_unscaled.csv",
    index=False
)

train_scaled.to_csv(
    OUT / "05_train_tabular_scaled.csv",
    index=False
)
val_scaled.to_csv(
    OUT / "06_validation_tabular_scaled.csv",
    index=False
)
test_scaled.to_csv(
    OUT / "07_test_tabular_scaled.csv",
    index=False
)

# ============================================================
# 12. AUDIT
# ============================================================

audit_rows = []

for split_name, part in [
    ("train", train),
    ("validation", val),
    ("test", test)
]:
    audit_rows.append({
        "split": split_name,
        "samples": len(part),
        "devices": part["uuid"].nunique(),
        "start": part["timestamp"].min(),
        "end": part["timestamp"].max(),
        "target_mean": part[TARGET].mean(),
        "target_std": part[TARGET].std()
    })

audit = pd.DataFrame(audit_rows)

audit.to_csv(
    OUT / "08_tabular_split_audit.csv",
    index=False
)

feature_manifest = pd.DataFrame([
    {
        "feature": f,
        "family":
            "current"
            if f in current_features
            else (
                "lag"
                if f in lag_features
                else "calendar"
            )
    }
    for f in FEATURES
])

feature_manifest.to_csv(
    OUT / "09_tabular_feature_manifest.csv",
    index=False
)

protocol = {
    "representation": "tabular",
    "target": TARGET,
    "forecast_horizon_minutes": 15,
    "history_minutes": N_LAGS * STEP_MINUTES,
    "number_of_lags": N_LAGS,
    "lag_resolution_minutes": STEP_MINUTES,
    "features": FEATURES,
    "feature_count": len(FEATURES),
    "uuid_used_as_predictor": False,
    "lag_crosses_segment": False,
    "partition_boundary_history_allowed": False,
    "scaler": "StandardScaler",
    "scaler_fit": "training predictors only",
    "target_scaled": False,
    "tree_models_can_use_unscaled_files": True,
    "ridge_can_use_scaled_files": True
}

with (
    OUT / "10_tabular_protocol.json"
).open("w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2)

# ============================================================
# 13. FINAL VALIDATION
# ============================================================

assert tabular.duplicated(
    subset=["uuid", "timestamp"]
).sum() == 0

assert tabular[FEATURES + [TARGET]].notna().all().all()

# Check train scaler has approximately zero mean after scaling.
scaled_train_means = (
    train_scaled[FEATURES].mean().abs()
)

assert scaled_train_means.max() < 1e-8

print("\n====================================")
print("FINAL TABULAR SPLIT")
print("====================================")
print(audit.to_string(index=False))

print("\n====================================")
print("FEATURES")
print("====================================")
for i, feature in enumerate(FEATURES, start=1):
    print(f"{i:02d}. {feature}")

print("\n====================================")
print("FINAL VALIDATION")
print("====================================")
print("✓ Lags remain inside contiguous segments")
print("✓ 1-hour history is exact (4 × 15 min)")
print("✓ Partition-boundary histories excluded")
print("✓ No missing predictors")
print("✓ UUID retained as metadata, not numeric predictor")
print("✓ StandardScaler fitted on TRAIN only")
print("✓ Target is not scaled")
print("✓ Unscaled files retained for RF/XGBoost")

print("\n====================================")
print("FILES CREATED")
print("====================================")
for f in sorted(OUT.iterdir()):
    print(" -", f.name)
